# Day 2 — LoRA fine-tuning (Colab)

S4DS KJSIT. **Post-break lab.** Runtime → Change runtime type → **GPU** (T4 is enough).

Goal: take a tiny instruct model, attach a LoRA adapter, do a short SFT pass so it
emits cleaner tool-call style replies — then see that *adapters*, not a full retrain,
did the work.

**You do not need this for Project 2.** Project 2 is MCP composition; this lab is
about when prompting/tools are not enough.


## 0. Install (GPU runtime)


In [ ]:
!pip install -q "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" "peft>=0.12.0" "trl>=0.11.0" "bitsandbytes>=0.43.0"

import torch
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — training will be slow; shorten MAX_STEPS below")


## 1. Why fine-tune at all? (read once)

| Approach | Use when |
|----------|----------|
| **Prompt** | Behaviour fits in the system message |
| **Tools / MCP** | Model needs fresh data or actions (Day 1–2 morning) |
| **Fine-tune** | Same format / style / domain every time, and prompts keep failing |

Function calling can be *prompted* (Day 1) or *trained* (this lab). Training
bakes the format into weights — fewer format failures, but costs GPU time and
you must re-train when the API schema changes a lot.


## 2. Tiny function-calling style dataset

We invent a small SFT set: user asks → assistant replies with a single tool call
block. Enough to *feel* SFT; not a production dataset.


In [ ]:
from datasets import Dataset

ROWS = [
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "What's the weather in Pune?"},
            {"role": "assistant", "content": "Thought: Need live weather.\nAction: get_weather(Pune)"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "Calculate 17 * 3 + 2"},
            {"role": "assistant", "content": "Thought: Arithmetic tool.\nAction: calculate(17 * 3 + 2)"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "Mess menu on Tuesday?"},
            {"role": "assistant", "content": "Thought: Need mess data.\nAction: get_mess_menu(Tuesday)"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "Is it raining in Mumbai?"},
            {"role": "assistant", "content": "Thought: Weather question.\nAction: get_weather(Mumbai)"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "What is 100 / 4?"},
            {"role": "assistant", "content": "Thought: Use calculator.\nAction: calculate(100 / 4)"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
            {"role": "user", "content": "Wednesday hostel food?"},
            {"role": "assistant", "content": "Thought: Mess menu.\nAction: get_mess_menu(Wednesday)"}
        ]
    },
]

# Repeat so the trainer has enough steps on a tiny set
train_ds = Dataset.from_list(ROWS * 20)
print(train_ds)
print(train_ds[0]["messages"][-1]["content"])


## 3. Load a small base model + LoRA

LoRA = low-rank adapters. Most weights stay frozen; we train a thin add-on.
QLoRA (optional) would quantise the base further — here plain LoRA on a 0.5B model
fits a free T4.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 4. Before training — sample a reply


In [ ]:
def generate(prompt_messages, max_new_tokens=80):
    prompt = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text

probe = [
    {"role": "system", "content": "You call tools using: Action: name(arg). Never invent Observations."},
    {"role": "user", "content": "Weather in Delhi?"},
]
print("BEFORE:", generate(probe))


## 5. SFT with TRL `SFTTrainer`

Short run on purpose — workshop tempo, not a leaderboard score.


In [ ]:
from trl import SFTConfig, SFTTrainer

def formatting(example):
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

MAX_STEPS = 30 if torch.cuda.is_available() else 5

args = SFTConfig(
    output_dir="qwen-lora-fc",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=MAX_STEPS,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
    packing=False,
)

# TRL moved kwargs around; try the modern signature first.
try:
    args.max_seq_length = 512
except Exception:
    pass

common = dict(
    model=model,
    args=args,
    train_dataset=train_ds,
    formatting_func=formatting,
)

try:
    trainer = SFTTrainer(**common, processing_class=tokenizer)
except TypeError:
    try:
        trainer = SFTTrainer(**common, tokenizer=tokenizer)
    except TypeError:
        trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tokenizer)

trainer.train()
print("train done")


## 6. After training — same probe


In [ ]:
print("AFTER:\n", generate(probe))
print()
print("Look for a clean Action: get_weather(Delhi) style line.")
print("If it is still messy: raise MAX_STEPS, or accept that 0.5B + 30 steps is a demo.")


## 7. What you should remember

1. **Prompt → tools → fine-tune** is a decision tree, not a ladder you always climb.
2. **LoRA** trains a small adapter; you can ship the adapter without copying the full model.
3. **SFT** teaches format/behaviour from demonstrations; alignment (RLHF/DPO) is a later stage.
4. Special tokens like `<tool_call>` on some models are just more characters in the string
   (Day 1 Lab 1) — training teaches the model to emit them reliably.

**Take-home:** `projects/project-2.md` — 2+ MCP servers. Building your own FastMCP is allowed there; we did not build servers in class.
